# TrOCR Apex Killfeed — Fine-tuning on Google Colab

**Steps before running:**
1. Run `python prepare_colab_bundle.py` locally → produces `colab_bundle.zip`
2. Upload `colab_bundle.zip` AND your model zip (e.g. `trocr_apex_20260311_0137.zip`) to Google Drive
3. In Colab: **Runtime → Change runtime type → T4 GPU**
4. Run cells top to bottom

In [ ]:
# ── Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── Cell 2: Extract colab_bundle.zip
import os, subprocess, zipfile, pathlib, glob

WORK_DIR = pathlib.Path('/content/apex_ocr')
WORK_DIR.mkdir(exist_ok=True)

result = subprocess.run(
    ['find', '/content/drive/MyDrive', '-name', 'colab_bundle.zip', '-type', 'f'],
    capture_output=True, text=True
)
matches = result.stdout.strip().splitlines()
if not matches:
    raise FileNotFoundError('colab_bundle.zip not found in Google Drive.')
BUNDLE_PATH = matches[0]
print(f'Found bundle: {BUNDLE_PATH}')

print('Extracting...')
with zipfile.ZipFile(BUNDLE_PATH) as zf:
    zf.extractall(WORK_DIR)

os.chdir(WORK_DIR)
print('Working directory:', os.getcwd())
print(f'Images: {len(glob.glob("crops/**/*.png", recursive=True))}')

import csv
csv.field_size_limit(10 * 1024 * 1024)
with open('labels/labels_clean.csv') as f:
    n = sum(1 for _ in f) - 1
print(f'Labels: {n}')

In [ ]:
# ── Cell 3: Extract checkpoint model from Drive
import subprocess, zipfile, pathlib

result = subprocess.run(
    ['find', '/content/drive/MyDrive', '-name', 'trocr_apex_*.zip', '-type', 'f'],
    capture_output=True, text=True
)
model_zips = sorted(result.stdout.strip().splitlines())
if not model_zips:
    print('No checkpoint zip found — will train from microsoft/trocr-small-printed')
    CHECKPOINT = 'microsoft/trocr-small-printed'
else:
    latest = model_zips[-1]
    print(f'Found checkpoint: {latest}')
    pathlib.Path('models').mkdir(exist_ok=True)
    with zipfile.ZipFile(latest) as zf:
        zf.extractall('models/')
    CHECKPOINT = 'models/trocr_apex'
    print(f'Extracted to {CHECKPOINT}')
    print('Files:', list(pathlib.Path(CHECKPOINT).iterdir()))

In [ ]:
# ── Cell 4: Install dependencies
!pip install -q transformers==4.40.0 timm sentencepiece
print('Done.')

In [ ]:
# ── Cell 5: Verify GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 6: Training config
import pathlib

EPOCHS = 15
BATCH_SIZE = 16
LR = 5e-5
QUALITY = {'confirmed', 'correction'}
BASE_MODEL = CHECKPOINT  # set by Cell 3
MODEL_OUT = pathlib.Path('models/trocr_apex')
NUM_WORKERS = 2

print(f'Epochs={EPOCHS}  Batch={BATCH_SIZE}  LR={LR}')
print(f'Quality={QUALITY}')
print(f'Base model: {BASE_MODEL}')

In [ ]:
# ── Cell 7: Dataset
import csv, random, pathlib
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

csv.field_size_limit(10 * 1024 * 1024)


class KillfeedDataset(Dataset):
    def __init__(self, rows, processor):
        self.rows = rows
        self.processor = processor

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        img = Image.open(row['filepath']).convert('RGB')
        pixel_values = self.processor(images=img, return_tensors='pt').pixel_values.squeeze(0)
        labels = self.processor.tokenizer(
            row['label'], padding='max_length', max_length=64,
            truncation=True, return_tensors='pt',
        ).input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels}


def load_rows(quality_filter):
    rows = []
    missing = 0
    with open('labels/labels_clean.csv', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if row['quality'] not in quality_filter:
                continue
            p = pathlib.Path(row['filepath'])
            if p.exists():
                rows.append(row)
            else:
                missing += 1
    if missing:
        print(f'  Warning: {missing} rows point to missing files')
    return rows


def stratified_split(rows, val_ratio=0.1, seed=42):
    rng = random.Random(seed)
    by_tier = {}
    for row in rows:
        by_tier.setdefault(row['quality'], []).append(row)
    train, val = [], []
    for tier_rows in by_tier.values():
        rng.shuffle(tier_rows)
        n_val = max(1, int(len(tier_rows) * val_ratio))
        val.extend(tier_rows[:n_val])
        train.extend(tier_rows[n_val:])
    rng.shuffle(train)
    rng.shuffle(val)
    return train, val


rows = load_rows(QUALITY)
train_rows, val_rows = stratified_split(rows)
print(f'Total: {len(rows)}  Train: {len(train_rows)}  Val: {len(val_rows)}')

In [ ]:
# ── Cell 8: Load model
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Loading {BASE_MODEL} ...')

processor = TrOCRProcessor.from_pretrained(BASE_MODEL)
model = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.to(device)
print(f'Model loaded on {device}')

In [ ]:
# ── Cell 9: Training loop
import shutil, tempfile
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup


def compute_cer(predictions, references):
    total_edits = total_chars = 0
    for pred, ref in zip(predictions, references):
        n, m = len(ref), len(pred)
        dp = list(range(m + 1))
        for i in range(1, n + 1):
            prev, dp[0] = dp[0], i
            for j in range(1, m + 1):
                prev, dp[j] = dp[j], prev if ref[i-1] == pred[j-1] else 1 + min(prev, dp[j], dp[j-1])
        total_edits += dp[m]
        total_chars += max(n, 1)
    return total_edits / total_chars if total_chars else 0.0


train_dataset = KillfeedDataset(train_rows, processor)
val_dataset   = KillfeedDataset(val_rows, processor)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

optimizer   = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps)

best_cer  = float('inf')
log_every = max(1, len(train_loader) // 20)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for i, batch in enumerate(train_loader, 1):
        pv  = batch['pixel_values'].to(device)
        lbl = batch['labels'].to(device)
        loss = model(pixel_values=pv, labels=lbl).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        train_loss += loss.item()
        if i % log_every == 0 or i == len(train_loader):
            print(f'  Ep {epoch}/{EPOCHS} | batch {i}/{len(train_loader)} | loss {train_loss/i:.4f}', flush=True)

    train_loss /= len(train_loader)

    model.eval()
    all_preds, all_refs = [], []
    with torch.no_grad():
        for batch in val_loader:
            pv  = batch['pixel_values'].to(device)
            lbl = batch['labels'].to(device)
            gen = model.generate(pv, max_new_tokens=64)
            all_preds.extend(processor.batch_decode(gen, skip_special_tokens=True))
            lbl[lbl == -100] = processor.tokenizer.pad_token_id
            all_refs.extend(processor.batch_decode(lbl, skip_special_tokens=True))

    cer = compute_cer(all_preds, all_refs)
    print(f'Epoch {epoch:02d}/{EPOCHS} | loss={train_loss:.4f} | CER={cer:.4f}')

    if cer < best_cer:
        best_cer = cer
        MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
        tmp = pathlib.Path(tempfile.mkdtemp(dir=MODEL_OUT.parent, prefix='.trocr_tmp_'))
        try:
            model.save_pretrained(tmp)
            processor.save_pretrained(tmp)
            if MODEL_OUT.exists():
                shutil.rmtree(MODEL_OUT)
            tmp.rename(MODEL_OUT)
        except Exception:
            shutil.rmtree(tmp, ignore_errors=True)
            raise
        print(f'  [BEST] CER={best_cer:.4f} → saved to {MODEL_OUT}')

print(f'\nDone. Best CER: {best_cer:.4f}')
if best_cer < 0.10:
    print('[OK] Target CER < 0.10 achieved.')
else:
    print('[WARN] CER >= 0.10 — consider more data or epochs.')

In [ ]:
# ── Cell 10: Save model to Google Drive
import shutil, datetime, pathlib

timestamp  = datetime.datetime.now().strftime('%Y%m%d_%H%M')
zip_name   = f'trocr_apex_{timestamp}'
zip_path   = pathlib.Path(f'/content/{zip_name}.zip')
drive_dest = pathlib.Path(f'/content/drive/MyDrive/{zip_name}.zip')

print(f'Zipping {MODEL_OUT} ...')
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', root_dir=MODEL_OUT.parent, base_dir=MODEL_OUT.name)
shutil.copy2(zip_path, drive_dest)

size_mb = drive_dest.stat().st_size / 1_048_576
print(f'Saved: {drive_dest}  ({size_mb:.1f} MB)')
print('Download and extract into your project as  models/trocr_apex/')